# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/helnagar123/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [29]:
import os

import duckdb
import pandas as pd

from google.colab import userdata
from huggingface_hub import login

In [30]:

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Login successful")

Login successful


In [31]:
con = duckdb.connect()

print("DuckDB Ready")

DuckDB Ready


In [32]:
from datasets import load_dataset

clients = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train"
)

clients

content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train"
)

print(content)
print(content.column_names)

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train"
)

print(daily)
print(daily.column_names)

query90 = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    split="train"
)

print(query90)
print(query90.column_names)


Dataset({
    features: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted'],
    num_rows: 519606
})
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 78835655
})
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chat

## 1. Unit of analysis + time window

One row represents one content–query pair for a rolling 90-day window.

Primary tables:
- dim_content
- fact_content_query_90d

Time window:
A mid-panel 90-day window (not the final June 2026 window).

Prediction / ranking target:
Rank content-query pairs by refresh opportunity using historical search performance.

Excluded:
Deleted content and future information after the decision point.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

### feature
search_volume
,competition
,word_count
,backlinks
,avg_position_90d
### Label
Label (proxy):
Historical search opportunity based on clicks_90d and impressions_90d.
### context
client_hash_id
,content_hash_id
,query_hash_id
,window_start
,window_end

### Excluded
is_deleted
,future data

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [36]:
content_arrow = content.data.table
daily_arrow = daily.data.table
query_arrow = query90.data.table

519606
78835655
2414248


In [37]:
con.register("dim_content", content_arrow)
con.register("fact_daily", daily_arrow)
con.register("fact_query90", query_arrow)
print("Tables registered successfully.")

Tables registered successfully.


In [38]:
con.sql("SHOW TABLES").df()

,name
0,dim_content
1,fact_daily
2,fact_query90


In [40]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    COUNT(
        DISTINCT content_hash_id || '-' ||
        query_hash_id || '-' ||
        CAST(window_start AS VARCHAR)
    ) AS unique_rows
FROM fact_query90
""").df()

,rows,unique_rows
0,2414248,2414248


In [41]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(window_start) AS first_window,
    MAX(window_end) AS last_window
FROM fact_query90
""").df()

,total_rows,first_window,last_window
0,2414248,2026-04-02,2026-06-30


In [42]:
con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM fact_daily
WHERE gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,28970051


In [43]:
features = con.sql("""
SELECT
    content_hash_id,
    query_hash_id,
    search_volume,
    competition,
    word_count,
    backlinks,
    impressions_90d,
    clicks_90d,
    avg_position_90d
FROM dim_content dc
JOIN fact_query90 fq
USING(content_hash_id, client_hash_id)
LIMIT 5
""").df()

features

,content_hash_id,query_hash_id,search_volume,competition,word_count,backlinks,impressions_90d,clicks_90d,avg_position_90d
0,content_47534b0d83d90e14,query_b58ba42948ad70f2,0,0.0,3095,0,11,0,64.454545
1,content_47605833fd625f6d,query_15dbefa7d7012d11,0,0.0,<NA>,0,14,0,6.214286
2,content_47605833fd625f6d,query_176aa63c11b78675,0,0.0,<NA>,0,26,0,2.000000
3,content_47605833fd625f6d,query_22be475a5525120f,0,0.0,<NA>,0,12,0,2.000000
4,content_47605833fd625f6d,query_2655ff86dbb53eda,0,0.0,<NA>,0,65,0,2.523077


| Feature          | Available when?                                                   |
| ---------------- | ----------------------------------------------------------------- |
| search_volume    | Historical keyword metric known before any refresh decision.      |
| competition      | Known before optimization because it comes from keyword research. |
| word_count       | Available from the existing content before editing.               |
| backlinks        | Existing page authority before the decision.                      |
| avg_position_90d | Computed from historical search performance before the refresh.   |


In [45]:
leak_df = con.sql("""
SELECT
    impressions_90d,
    clicks_90d,
    avg_position_90d,
    clicks_90d AS leaky_feature
FROM fact_query90
LIMIT 10000
""").df()

leak_df.head()

,impressions_90d,clicks_90d,avg_position_90d,leaky_feature
0,11,0,10.818182,0
1,13,0,1.769231,0
2,16,0,23.562500,0
3,55,0,2.200000,0
4,14,0,3.428571,0


Leakage demonstration: leaky_feature is directly derived from the target (clicks_90d). Including it would produce unrealistically high model performance, so it must be removed before training.

## 4. Data limits

- Rolling 90-day windows overlap and are not independent.
- Historical coverage is uneven across clients because data availability differs.
- Some rows may have missing GSC/GA4 data.
- Future refresh outcomes cannot be inferred from this dataset alone.
- Pseudonymized IDs prevent page- or client-level interpretation.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.